In [55]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')


In [56]:
# For Plotly theme

PLOTLY_LAYOUT = dict(
    template      = 'plotly_dark',
    paper_bgcolor = '#1a1a2e',
    plot_bgcolor  = '#1a1a2e',
    font          = dict(family='monospace', color='#ccc'),
    margin        = dict(t=60, b=40, l=40, r=40),
)

PURPLE = '#7c6fcd'
GREEN  = '#4ecb8d'
ORANGE = '#f4845f'

print('Libraries ready')

Libraries ready


In [57]:
df = pd.read_csv("/content/data.csv", encoding = 'latin1')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [59]:
print(f'Shape:      {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns:    {df.columns.tolist()}')

print(f'Countries:  {df["Country"].nunique()} unique')
df.head()

Shape:      541,909 rows × 8 columns
Columns:    ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
Countries:  38 unique


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


## Data Cleaning and Transformation


In [60]:
df.isnull().sum()

,0
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


1. Drop rows with no `CustomerID` — cannot track anonymous buyers
2. Remove negative/zero quantities and prices — errors or refunds

In [61]:
df[df['CustomerID'].isnull()]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,12/1/2010 11:52,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,12/1/2010 14:32,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,12/1/2010 14:32,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,12/1/2010 14:32,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,12/1/2010 14:32,1.66,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
541536,581498,85099B,JUMBO BAG RED RETROSPOT,5,12/9/2011 10:26,4.13,NaN,United Kingdom
541537,581498,85099C,JUMBO BAG BAROQUE BLACK WHITE,4,12/9/2011 10:26,4.13,NaN,United Kingdom
541538,581498,85150,LADIES & GENTLEMEN METAL SIGN,1,12/9/2011 10:26,4.96,NaN,United Kingdom
541539,581498,85174,S/4 CACTI CANDLES,1,12/9/2011 10:26,10.79,NaN,United Kingdom


In [62]:
duplicate_rows = df.duplicated().sum()
duplicate_rows

np.int64(5268)

In [63]:
duplicate_rows = df[df.duplicated(keep=False)]
display(duplicate_rows.sort_values(by=['InvoiceNo','StockCode']))

print('CustomerID is missing for 24.9% of rows — anonymous/guest buyers.')
print('We drop them because we cannot track a customer without an ID.')

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,12/1/2010 11:45,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,12/1/2010 11:45,1.25,17908.0,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,12/1/2010 11:45,4.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,12/1/2010 11:45,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,12/1/2010 11:45,2.10,17908.0,United Kingdom
...,...,...,...,...,...,...,...,...
440149,C574510,22360,GLASS JAR ENGLISH CONFECTIONERY,-1,11/4/2011 13:25,2.95,15110.0,United Kingdom
461407,C575940,23309,SET OF 60 I LOVE LONDON CAKE CASES,-24,11/13/2011 11:38,0.55,17838.0,United Kingdom
461408,C575940,23309,SET OF 60 I LOVE LONDON CAKE CASES,-24,11/13/2011 11:38,0.55,17838.0,United Kingdom
529980,C580764,22667,RECIPE BOX RETROSPOT,-12,12/6/2011 10:38,2.95,14562.0,United Kingdom


CustomerID is missing for 24.9% of rows — anonymous/guest buyers.
We drop them because we cannot track a customer without an ID.


In [64]:
# drop the missing customer id
df = df.dropna(subset=["CustomerID"]).copy()

# drop the cancelled orders
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [65]:
df['Revenue'] = df['Quantity']*df['UnitPrice']  # create revenue Column

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['DayOfWeek']  = df['InvoiceDate'].dt.day_name()
df['Hour']       = df['InvoiceDate'].dt.hour
df['YearMonth']  = df['InvoiceDate'].dt.to_period('M').astype(str)


df['CustomerID']= df['CustomerID'].astype(int)

In [66]:
print(f"Clean shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Unique customers: {df['CustomerID'].nunique():,}")

Clean shape: 397,884 rows × 12 columns
Unique customers: 4,338


## **EXPLORATORY** **DATA** **ANALYSIS**

**Monthly revenue Trend**

In [68]:
monthly = df.groupby('YearMonth')['Revenue'].sum().reset_index()
monthly.columns = ['Month', 'Revenue']

fig = go.Figure()

fig.add_trace(go.Scatter(
    x            = monthly['Month'],
    y            = monthly['Revenue'],
    mode         = 'lines+markers',
    line         = dict(color=PURPLE, width=2.5),
    marker       = dict(size=7, color=PURPLE),
    fill         = 'tozeroy',
    fillcolor    = 'rgba(124,111,205,0.15)',
    hovertemplate = '<b>%{x}</b><br>Revenue: £%{y:,.0f}<extra></extra>',
))

fig.update_layout(
    **PLOTLY_LAYOUT,
    title  = dict(text='Monthly revenue trend', font=dict(size=16)),
    xaxis  = dict(title='Month', tickangle=45, gridcolor='#2a2a38'),
    yaxis  = dict(title='Revenue (£)', gridcolor='#2a2a38',
                  tickformat='£,.0f'),
    height = 420,
)

fig.show()
peak = monthly.loc[monthly['Revenue'].idxmax()]
print(f'Peak month: {peak["Month"]} — £{peak["Revenue"]:,.0f}')

Peak month: 2011-11 — £1,161,817


**Revenue by Country (top 10)**

In [70]:
country_rev = (
    df.groupby('Country')['Revenue']
    .sum()
    .reset_index()
    .sort_values('Revenue', ascending=False)
    .head(10)
)
country_rev['Share_%'] = (country_rev['Revenue'] / country_rev['Revenue'].sum() * 100).round(1)
country_rev['Color']   = country_rev['Country'].apply(
    lambda c: GREEN if c == 'United Kingdom' else PURPLE
)

fig = go.Figure(go.Bar(
    x            = country_rev['Revenue'],
    y            = country_rev['Country'],
    orientation  = 'h',
    marker_color = country_rev['Color'],
    text         = country_rev['Revenue'].apply(lambda x: f'£{x/1000:.0f}K'),
    textposition = 'outside',
    hovertemplate = (
        '<b>%{y}</b><br>'
        'Revenue: £%{x:,.0f}<br>'
        'Share: %{customdata:.1f}%<extra></extra>'
    ),
    customdata   = country_rev['Share_%'],
))

fig.update_layout(
    **PLOTLY_LAYOUT,
    title      = dict(text='Top 10 countries by revenue', font=dict(size=16)),
    xaxis      = dict(title='Revenue (£)', gridcolor='#2a2a38', tickformat='£,.0f'),
    yaxis      = dict(title='', autorange='reversed'),
    height     = 420,
    showlegend = False,
)

fig.show()
uk_share = country_rev.loc[country_rev['Country']=='United Kingdom', 'Share_%'].values[0]
print(f'UK accounts for {uk_share}% of all revenue in the top 10 countries.')

UK accounts for 84.7% of all revenue in the top 10 countries.


**Orders by day of the week**

In [73]:
day_order  = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
orders_day = (
    df.groupby('DayOfWeek')['InvoiceNo']
    .nunique()
    .reindex(day_order)
    .reset_index()
)
orders_day.columns = ['Day', 'Orders']
orders_day['Color'] = orders_day['Day'].apply(
    lambda d: ORANGE if d == 'Thursday' else PURPLE
)

fig = go.Figure(go.Bar(
    x            = orders_day['Day'],
    y            = orders_day['Orders'],
    marker_color = orders_day['Color'],
    text         = orders_day['Orders'].apply(lambda x: f'{x:,}'),
    textposition = 'outside',
    hovertemplate = '<b>%{x}</b><br>Orders: %{y:,}<extra></extra>',
))

fig.update_layout(
    **PLOTLY_LAYOUT,
    title  = dict(text='Number of orders by day of week', font=dict(size=16)),
    xaxis  = dict(title=''),
    yaxis  = dict(title='Unique orders', gridcolor='#2a2a38'),
    height = 400,
    showlegend = False,
)

fig.show()
print('Thursday is the busiest day. Saturday has almost zero orders.')

Thursday is the busiest day. Saturday has almost zero orders.


**Customer Revenue Distribution (Log scale)**

Log scale is used because a few customers spend £50,000+ while most spend under £500.
Without log scale, most bars would be invisible.

In [77]:
customer_rev = df.groupby('CustomerID')['Revenue'].sum().reset_index()
customer_rev.columns = ['CustomerID', 'Total_Revenue']
customer_rev['Log_Revenue'] = np.log1p(customer_rev['Total_Revenue'])

fig = go.Figure(go.Histogram(
    x             = customer_rev['Log_Revenue'],
    nbinsx        = 50,
    marker_color  = PURPLE,
    opacity       = 0.85,
    hovertemplate = 'log(revenue) bucket: %{x:.1f}<br>Customers: %{y:,}<extra></extra>',
))

median_log = np.log1p(customer_rev['Total_Revenue'].median())
fig.add_vline(
    x             = median_log,
    line_dash     = 'dash',
    line_color    = GREEN,
    annotation_text = f'Median £{customer_rev["Total_Revenue"].median():.0f}',
    annotation_position = 'top right',
)

fig.update_layout(
    **PLOTLY_LAYOUT,
    title  = dict(text='Customer revenue distribution (log scale)', font=dict(size=16)),
    xaxis  = dict(title='log(Total Revenue + 1) — each step = 10x more revenue', gridcolor='#2a2a38'),
    yaxis  = dict(title='Number of customers', gridcolor='#2a2a38'),
    height = 400,
)

fig.show()
print(f' Median customer revenue: £{customer_rev["Total_Revenue"].median():.0f}')
print(f' Top customer revenue:    £{customer_rev["Total_Revenue"].max():,.0f}')

 Median customer revenue: £674
 Top customer revenue:    £280,206


**Order  activity Heatmap (Day x hour)**

In [80]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

heatmap_df = (
    df.groupby(['DayOfWeek','Hour'])['InvoiceNo']
    .nunique()
    .reset_index()
)
heatmap_df.columns = ['Day', 'Hour', 'Orders']

pivot = (
    heatmap_df.pivot(index='Day', columns='Hour', values='Orders')
    .reindex(day_order)
    .fillna(0)
)

fig = go.Figure(go.Heatmap(
    z             = pivot.values,
    x             = [f'{h:02d}:00' for h in pivot.columns],
    y             = pivot.index.tolist(),
    colorscale    = 'YlOrRd',
    hovertemplate = '<b>%{y}</b> at <b>%{x}</b><br>Orders: %{z:,}<extra></extra>',
    colorbar      = dict(title='Orders'),
))

fig.update_layout(
    **PLOTLY_LAYOUT,
    title  = dict(text='Order activity heatmap — day × hour', font=dict(size=16)),
    xaxis  = dict(title='Hour of day'),
    yaxis  = dict(title=''),
    height = 380,
)

fig.show()
print('Peak activity: Tuesday–Thursday between 10am and 2pm.')

Peak activity: Tuesday–Thursday between 10am and 2pm.


**Top 10 products by revenue**

In [81]:
top_products = (
    df.groupby('Description')['Revenue']
    .sum()
    .reset_index()
    .sort_values('Revenue', ascending=False)
    .head(10)
)
top_products.columns = ['Product', 'Revenue']

# Short label for display, full name in hover
top_products['Short'] = top_products['Product'].apply(
    lambda s: s[:30] + '…' if len(s) > 30 else s
)

fig = go.Figure(go.Bar(
    x            = top_products['Revenue'],
    y            = top_products['Short'],
    orientation  = 'h',
    marker_color = ORANGE,
    text         = top_products['Revenue'].apply(lambda x: f'£{x/1000:.1f}K'),
    textposition = 'outside',
    hovertemplate = (
        '<b>%{customdata}</b><br>'
        'Revenue: £%{x:,.0f}<extra></extra>'
    ),
    customdata   = top_products['Product'],
))

fig.update_layout(
    **PLOTLY_LAYOUT,
    title      = dict(text='Top 10 products by revenue', font=dict(size=16)),
    xaxis      = dict(title='Revenue (£)', gridcolor='#2a2a38', tickformat='£,.0f'),
    yaxis      = dict(title='', autorange='reversed'),
    height     = 430,
    showlegend = False,
)

fig.show()

# customer summary table

In [83]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f'Snapshot date (our "today"): {snapshot_date.date()}')

customer_summary = df.groupby('CustomerID').agg(
    first_purchase  = ('InvoiceDate', 'min'),
    last_purchase   = ('InvoiceDate', 'max'),
    total_orders    = ('InvoiceNo',   'nunique'),
    total_items     = ('Quantity',    'sum'),
    total_revenue   = ('Revenue',     'sum'),
    avg_order_value = ('Revenue',     'mean'),
    unique_products = ('StockCode',   'nunique'),
).reset_index()

customer_summary['recency_days'] = (
    snapshot_date - customer_summary['last_purchase']
).dt.days

customer_summary['tenure_days'] = (
    snapshot_date - customer_summary['first_purchase']
).dt.days

print(f'   Rows:    {customer_summary.shape[0]:,} customers')
print(f'   Columns: {customer_summary.shape[1]}')
customer_summary.head()

Snapshot date (our "today"): 2011-12-10
   Rows:    4,338 customers
   Columns: 10


,CustomerID,first_purchase,last_purchase,total_orders,total_items,total_revenue,avg_order_value,unique_products,recency_days,tenure_days
0,12346,2011-01-18 10:01:00,2011-01-18 10:01:00,1,74215,77183.60,77183.600000,1,326,326
1,12347,2010-12-07 14:57:00,2011-12-07 15:52:00,7,2458,4310.00,23.681319,103,2,367
2,12348,2010-12-16 19:09:00,2011-09-25 13:13:00,4,2341,1797.24,57.975484,22,75,358
3,12349,2011-11-21 09:51:00,2011-11-21 09:51:00,1,631,1757.55,24.076027,73,19,19
4,12350,2011-02-02 16:01:00,2011-02-02 16:01:00,1,197,334.40,19.670588,17,310,310


## Recency vs revenue scatter

Customers on the right (high recency) are likely churners.

In [85]:
sample = customer_summary.sample(min(1500, len(customer_summary)), random_state=42).copy()
sample['log_revenue'] = np.log1p(sample['total_revenue'])

fig = px.scatter(
    sample,
    x              = 'recency_days',
    y              = 'log_revenue',
    color          = 'total_orders',
    color_continuous_scale = 'Plasma',
    size           = 'total_orders',
    size_max       = 14,
    opacity        = 0.65,
    hover_data     = {
        'CustomerID':    True,
        'total_revenue': ':,.0f',
        'total_orders':  True,
        'recency_days':  True,
        'log_revenue':   False,
    },
    labels         = {
        'recency_days': 'Recency (days since last purchase)',
        'log_revenue':  'log(Total Revenue + 1)',
        'total_orders': 'Orders',
    },
)

fig.update_layout(
    **PLOTLY_LAYOUT,
    title  = dict(text='Recency vs revenue — each dot is one customer', font=dict(size=16)),
    xaxis  = dict(gridcolor='#2a2a38'),
    yaxis  = dict(gridcolor='#2a2a38'),
    height = 460,
    coloraxis_colorbar = dict(title='Orders'),
)

fig.show()
print('Customers far to the RIGHT (high recency) = likely churners.')
print('Bright dots (high order count) tend to cluster on the LEFT = loyal customers.')

Customers far to the RIGHT (high recency) = likely churners.
Bright dots (high order count) tend to cluster on the LEFT = loyal customers.


## Monthly new vs returning customers
New customers = first purchase in that month.

Returning = bought before.

In [86]:
# Get each customer's first purchase month
first_months = (
    df.groupby('CustomerID')['YearMonth']
    .min()
    .reset_index()
)
first_months.columns = ['CustomerID', 'FirstMonth']

# Tag each transaction as new or returning
df_tagged = df.merge(first_months, on='CustomerID')
df_tagged['CustomerType'] = df_tagged.apply(
    lambda r: 'New' if r['YearMonth'] == r['FirstMonth'] else 'Returning', axis=1
)

monthly_cust = (
    df_tagged.groupby(['YearMonth','CustomerType'])['CustomerID']
    .nunique()
    .reset_index()
)
monthly_cust.columns = ['Month', 'Type', 'Customers']

fig = px.bar(
    monthly_cust,
    x             = 'Month',
    y             = 'Customers',
    color         = 'Type',
    color_discrete_map = {'New': ORANGE, 'Returning': PURPLE},
    barmode       = 'stack',
    text_auto     = True,
    labels        = {'Customers': 'Unique customers', 'Month': ''},
)

fig.update_traces(
    hovertemplate = '<b>%{x}</b><br>%{fullData.name}: %{y:,}<extra></extra>',
    textfont_size = 9,
)

fig.update_layout(
    **PLOTLY_LAYOUT,
    title  = dict(text='Monthly new vs returning customers', font=dict(size=16)),
    xaxis  = dict(tickangle=45, gridcolor='#2a2a38'),
    yaxis  = dict(gridcolor='#2a2a38'),
    height = 430,
    legend = dict(title='Customer type'),
)

fig.show()
print('Orange = brand new customers. Purple = customers who bought before.')
print('A healthy business grows its returning base month over month.')

Orange = brand new customers. Purple = customers who bought before.
A healthy business grows its returning base month over month.


In [87]:
customer_summary.to_csv("customer_summary.csv", index=False)

In [88]:
print(f"Saved customer_summary.csv  ({len(customer_summary):,} customers)")

Saved customer_summary.csv  (4,338 customers)
